# Combine Datasets

In [1]:
import os
import pandas as pd

base_dir = os.getcwd()
data_dir = os.path.join(base_dir, 'Ausgrid_solar_home_data')

# Load processed datasets
ausgrid = pd.read_csv(os.path.join(data_dir, "ausgrid_processed.csv"))
weather = pd.read_csv(os.path.join(data_dir, "weather_data_processed.csv"))

# Combine datasets
df = ausgrid.merge(weather, on="datetime", how="left")
df.head()

,Customer,Generator Capacity,Postcode,Consumption Category,datetime,kWh,Unnamed: 0,temperature_2m,shortwave_radiation,precipitation
0,1,3.78,2076,CL,2010-07-01 00:00:00,1.075,0.0,3.60,0.0,0.0
1,1,3.78,2076,CL,2010-07-01 00:30:00,1.250,1.0,3.45,0.0,0.0
2,1,3.78,2076,CL,2010-07-01 01:00:00,1.244,2.0,3.30,0.0,0.0
3,1,3.78,2076,CL,2010-07-01 01:30:00,1.256,3.0,2.75,0.0,0.0
4,1,3.78,2076,CL,2010-07-01 02:00:00,0.744,4.0,2.20,0.0,0.0


The Ausgrid dataset uses three consumption categories:
- GC: General Consumption
- CL: Controlled Load
- GG: Gross Generation (solar PV generation)

Consider a real distribution system. Net load is monitored for each customer, but residential solar generation may not be directly measured. The size, orientation, and other characteristics of the solar installations may also be unknown.
Our goal is to disaggregate net load into base load and PV generation. Net load is given by $P_\text{net} = P_\text{load} - P_\text{PV}$ where $P_\text{load} = GC + CL$.
We can do this by predicting $P_\text{PV}$.

A common simplified equation for PV power output is:

$P_\text{PV} = P_\text{rated} \frac{G}{G_\text{STC}}[1 + \gamma (T_\text{cell} - T_\text{STC})]$
- $P_\text{PV}$ = PV electrical power output (kW).
- $P_\text{rated}$ = rated/nominal PV system capacity (kW).
- G = indicent solar irradiance on the panels (W/m^2).
- $G_\text{STC}$ = reference irradiance at standard test conditions, typically 1000 W/m^2.
- $T_\text{cell}$ = PV cell temperature (C)
- $T_\text{STC}$ = reference cell temperature, typically 25 C.
- $\gamma$ = temperature coefficient of power (per C), normally negative.

Irradiance primarily determines how much power is available, where cell temperature modifies the output. Solar irradiance is the solar power received per unit area, typically measured in W/m^2. For PV modeling, it ideally refers to the radiation incident on the panel surface. Shortwave radiation is a broader atmospheric measurement of incoming radiation in the shortwave portion of the spectrum, also measured in W/m^2. In our dataset, `shortwave_radiation` can serve as a proxy for the solar irradiance available to the PV system.

The rated PV system capacity $P_\text{rated}$ is unknown. The incident solar radiation $G$ can be estimated from `shortwave_radiation` and the position of the sun, which can in turn be estimated from the `hour of day` and the `geographical location` of the system. A useful starting point to to model the irradiance incident on the PV panels as:

$G(t) = \alpha S(t) f(\theta_\text{sun}(t, \phi, \lambda), \beta)$
- $G(t)$ = irradiance incident on the PV panels at time t (W/m^2)
- $S(t)$ = measured shortwave radiation (W/m^2)
- t = time of day/date.
- $\phi, \lambda$ = latitude and longitude of the PV system.
- $\theta_\text{sun}$ = solar position.
- $\beta$ = panel tilt, which is unknown and must be estimated.
- $f(\dot)$ = function describing how solar position and panel orientation determine the radiation incident on the panel.
- $\alpha$ = effective scaling / correction parameter.

$\theta_\text{sun} = \sin \phi \sin \delta + \cos \phi \cos \delta \cos H$
- $\delta$ = solar declination, determined by the date.
- $H$ = solar hour angle, determined by time of day.

$\delta = 23.45 \degree \sin (\frac{360 \degree}{365}(284 + n))$
- $n$ = day of year (1-365)

The temperature coefficient of power $\gamma$, panel tilt parameter $\beta$, and correction parameter $\alpha$ must be estimated, all other information is known. We can combine all multiplied unknowns into one unknown parameter $C = P_\text{rated} \times \beta \times \alpha$. 

## Calculate Sun Position

In [ ]:
import numpy as np
import pandas as pd

lat = 33.9 # system latitude (degrees)
lon = 151.2 # system longitude (degrees)

# Convert string to datetime type
df["datetime"] = pd.to_datetime(df["datetime"])

sun = pd.DataFrame({"datetime": df["datetime"]})
sun["day_of_year"] = sun["datetime"].dt.dayofyear
sun["hour"] = sun["datetime"].dt.hour + sun["datetime"].dt.minute / 60

# Solar declination
sun["declination_angle"] = np.radians(23.45 * np.sin(np.radians((360 / 365) * (284 + sun["day_of_year"]))))

# Solar hour angle (simplified: 15° per hour from solar noon)
hour_angle = np.radians(15 * (sun['hour'] - 12))

# Solar zenith angle
lat_rad = np.radians(lat)
sun["sun_position"] = np.degrees(np.arccos(
    np.sin(lat_rad) * np.sin(sun["declination_angle"]) +
    np.cos(lat_rad) * np.cos(sun["declination_angle"]) * np.cos(hour_angle)))

# Make sure sun does not have duplicate datetime values
sun = sun[["datetime", "sun_position"]].drop_duplicates("datetime")

sun.head()

### Append sun position to original dataset

In [ ]:
df = df.merge(
    sun[["datetime", "sun_position"]],
    on="datetime",
    how="left")

df.head()

Our model is simplified to:
$P_\text{PV}(t) = C \times S(t) \frac{1 + \gamma (T_\text{Cell}(t) - T_\text{STC})}{G_\text{STC}}$
- $S(t)$ is `shortwave_radiation`.
- $T_\text{cell}(t)$ is estimated as `temperature_2m`.
- $T_\text{STC}$ and $G_\text{STC}$ are known constants.
- $C$ is a per-customer unknown parameter accounting for PV system characteristics such as total generating capacity and panel tilt angle.
- $\gamma$ is a per-customer unknown parameter representing the PV system's temperature coefficient of power (per C).

# Parameter Estimation
## Modify the dataset

In [ ]:
df["Pload"] = df["kWh"].where(df["Consumption Category"].isin(["GC", "CL"]), 0)
df["PPV"]   = df["kWh"].where(df["Consumption Category"] == "GG", 0)
df["Pnet"]  = df["Pload"] - df["PPV"]
df.drop(columns=["Consumption Category", "kWh"], inplace=True)
df.head()